# 02 — LoRA instruction fine-tune (Qwen3-VL-8B)

Rung 02. Baseline = `00-baseline` (zero-shot, raw 0.262). **The ONE variable: LoRA ON.** Frame sampling, system prompt, judge, generation all byte-identical to rung 00.

Split = frozen `experiments/splits/frame_ood_v1.csv` (train 92vid/13748q · val_id chole=ID · val_ood Sigmoid=OOD). Target the weak formats **fo_class + number**. Select checkpoint by **acc_OOD**. Runs on a GPU pod (A100 80GB, volume gf78k60nlt).

In [ ]:
# ── bootstrap ─────────────────────────────────────────
import sys, logging
from pathlib import Path
EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / '.git').exists() or (REPO / 'src').is_dir()):
    REPO = REPO.parent
for p in (EXP_DIR / '_models', REPO / 'src'):
    if p.is_dir():
        sys.path.insert(0, str(p))
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s %(message)s', datefmt='%H:%M:%S')
import lora_sft_train as engine
from frame.config import BaselineConfig
from frame.data import load_frame_items
from frame import split as sp
from frame import delta as dl
from frame.run import run_baseline

## Sibling rungs (change ONE value, rerun, bump run_name)
- 02a: `learning_rate=1e-4` (if underfit) · 02b: `lora_rank=16` · 02c: epoch sweep · 02d: oversample fo_class+number rows.
Each = prose + a new `runs/<tag>/` + a RESULTS.csv row — NOT a new notebook.

In [ ]:
# ── config (inline) ───────────────────────────────────
SMOKE = True   # START True: tiny end-to-end pass (16 ex, 2 steps). Flip to False for the full run.
cfg = engine.LoRAConfig(
    data_root     = Path('/workspace/orena-data'),
    model_path    = Path('/workspace/models/qwen3-vl-8b'),
    exp_dir       = EXP_DIR,
    manifest_path = REPO / 'experiments' / 'splits' / 'frame_ood_v1.csv',
    run_name      = '02_lora_sft_v1' if not SMOKE else '02_lora_sft_smoke',
    lora_rank=8, lora_alpha=32, lora_dropout=0.1, learning_rate=2e-5, num_train_epochs=5,
    smoke=SMOKE,
)
cfg.run_dir.mkdir(parents=True, exist_ok=True)
print('run_dir:', cfg.run_dir, '| SMOKE:', SMOKE)

In [ ]:
# ── stage 1: export train JSONL (frames + ShareGPT) ───
engine.main(cfg, 'export')
print('train.jsonl:', cfg.train_jsonl, '(', sum(1 for _ in open(cfg.train_jsonl)), 'examples )')

In [ ]:
# ── stage 2: LoRA train (ms-swift) ────────────────────
# Full run: execute this notebook headless so it survives session death:
#   jupyter nbconvert --to notebook --execute 02_lora_sft.ipynb --ExecutePreprocessor.timeout=-1
engine.main(cfg, 'train')

In [ ]:
# ── training loss curve (image) ───────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, json, glob
logs = sorted(glob.glob(str(cfg.ckpt_dir / '**' / 'logging.jsonl'), recursive=True))
if logs:
    rec = [json.loads(l) for l in open(logs[-1]) if l.strip()]
    steps = [r.get('global_step') for r in rec if 'loss' in r]
    loss  = [r['loss'] for r in rec if 'loss' in r]
    if loss:
        fig, ax = plt.subplots(figsize=(8,4)); ax.plot(steps, loss); ax.set_xlabel('step'); ax.set_ylabel('train loss'); ax.set_title('LoRA training loss')
        fig.tight_layout(); fig.savefig(cfg.run_dir / 'loss_curve.png', dpi=130); plt.close(fig)
        print('saved', cfg.run_dir / 'loss_curve.png')
else:
    print('no logging.jsonl yet (check ms-swift log path under', cfg.ckpt_dir, ')')

In [ ]:
# ── stage 3: per-epoch checkpoint selection by acc_OOD (CONSTITUTION §IV.2) ──
# Merge EACH epoch, eval it on the OOD (Sigmoid) videos only (cheap), pick the best.
# Past epoch 1-2 risks OOD collapse (PITFALLS #1) — never default to the last epoch.
import pandas as pd
vs = sp.load_manifest(cfg.manifest_path)
ood_videos = {k for k, s in vs.items() if s == 'val_ood'}
ckpts = engine.list_checkpoints(cfg)
print(f'{len(ckpts)} checkpoints; selecting by acc_OOD on {len(ood_videos)} Sigmoid videos')
sel = []
for ck in ckpts:
    merged = engine.merge_checkpoint(cfg, ck)
    ecfg = BaselineConfig(data_root=cfg.data_root, model_path=merged,
                          out_dir=cfg.run_dir / 'sel', run_name='ood_' + ck.name,
                          max_pixels=cfg.max_pixels, seed=cfg.seed,
                          n_eval=(32 if SMOKE else None))
    rep = run_baseline(ecfg, video_filter=ood_videos)
    acc = rep['raw_accuracy']
    sel.append({'ckpt': ck.name, 'path': str(merged), 'acc_ood': acc})
    print(f'  {ck.name}: acc_OOD(raw) = {acc:.4f}')
seldf = pd.DataFrame(sel)
seldf.to_csv(cfg.run_dir / 'checkpoint_selection.csv', index=False)
best = seldf.loc[seldf.acc_ood.idxmax()]
best_merged = Path(best['path'])
best_acc = best['acc_ood']
print(f'BEST by acc_OOD: {best.ckpt}  acc_OOD={best_acc:.4f}')

In [ ]:
# ── stage 4: FULL eval of the OOD-selected checkpoint (val_id + val_ood) ──
# Reuse the baseline harness with model_path -> the best merged model. Identical prompt/frames/judge.
eval_cfg = BaselineConfig(
    data_root=cfg.data_root, model_path=best_merged,
    out_dir=cfg.run_dir, run_name='eval_best',
    max_pixels=cfg.max_pixels, seed=cfg.seed,
    n_eval=(64 if SMOKE else None),
)
lora_report = run_baseline(eval_cfg)
print('BEST LoRA pre_evaluation_score:', lora_report['pre_evaluation_score'], '| raw:', lora_report['raw_accuracy'])

In [ ]:
# ── stage 5: Δ vs zero-shot — WHICH question types improved ──
import pandas as pd
# zero-shot arm = the 00-baseline run on the SAME test set (val). Point at its results.csv.
ZS = REPO / 'experiments' / '00-baseline' / 'runs' / '00_zeroshot_qwen3vl' / 'results.csv'
LO = cfg.run_dir / 'eval_best' / 'results.csv'      # the OOD-selected checkpoint's full-val eval
assert ZS.exists(), f'zero-shot results.csv missing at {ZS} — re-run 00-baseline on the test set first'
zs_df, lo_df = pd.read_csv(ZS), pd.read_csv(LO)
# metadata (answer_format, capability_group, ID/OOD from the manifest)
items = load_frame_items(BaselineConfig(data_root=cfg.data_root), splits=('test',))
vs = sp.load_manifest(cfg.manifest_path)
meta = dl.question_metadata(items, video_split=vs)
report = dl.delta_report(zs_df, lo_df, meta)
for k, df in report.items():
    print('\n===', k, '===\n', df.to_string(index=False))
dl.write_delta_csv(report, cfg.run_dir / 'RESULTS_delta.csv')
dl.plot_delta(report, cfg.run_dir / 'delta_by_format.png')
print('\nwrote', cfg.run_dir / 'RESULTS_delta.csv', '+ delta_by_format.png')

## Result
Fill the ladder in README + `RESULTS.csv` from `headline` (acc_ID / acc_OOD / Δ) and the per-type Δ in `runs/<tag>/RESULTS_delta.csv`. **Verdict rule:** GO only if **acc_OOD ↑ vs zero-shot** and no format regressed; select the epoch that maximizes acc_OOD (Sigmoid). Images: `loss_curve.png`, `delta_by_format.png`. For the submission model, retrain on train+val (all 20k) with the winning recipe.